
# 🚕 NYC YELLOW TAXI DAILY TRIP ANALYTICS

## Overview

Explore key daily metrics for New York City's Yellow Taxi trips, uncovering ridership trends and revenue performance in style.

---

### 📊 **Aggregated Daily Metrics**

- **Total Trips:** Number of trips completed each day.
- **Average Passengers:** Average number of passengers per trip.
- **Average Distance:** Typical journey length in miles.
- **Average Fare per Trip:** Mean fare collected for each trip.
- **Revenue:** Total daily earnings.
- **Max/Min Fare:** Extremes in trip pricing.

---

<div style="font-size: 90%; color: #888;">
Data from: <b>NYC Taxi - Silver/Gold Tables</b>  
Updated: <b>August 2026</b>
</div>

---

In [0]:
# =====================================================
# CALCULATE START TIME
# =====================================================
from datetime import datetime
load_start_time = datetime.now()

In [0]:
from pyspark.sql.functions import *
from datetime import datetime

In [0]:
enriched_yellow_taxi_df = spark.read.table("NYCTAXI.SILVER.YELLOW_TAXI_TRIPS_ENRICHED")

In [0]:
analytics_df = (enriched_yellow_taxi_df.
        # group records by calendar date
        groupBy(enriched_yellow_taxi_df.tpep_pickup_datetime.cast("date").alias("pickup_date") ).
        agg(
            count("*").alias("total_trips"),                             # total number of trips per day
            round(avg("passenger_count"), 1).alias("average_passengers"), # average passengers per trip
            round(avg("trip_distance"), 1).alias("average_distance"),     # average trip distance (miles)
            round(avg("fare_amount"), 2).alias("average_fare_per_trip"),   # average fare per trip ($)
            max("fare_amount").alias("max_fare"),                         # highest single-trip fare
            min("fare_amount").alias("min_fare"),                         # lowest single-trip fare
            round(sum("total_amount"), 2).alias("total_revenue")          # total revenue for the day ($)
        )
        )

#### LOAD DAILY TRIPS SUMMARY
- `NYCTAXI.GOLD.YELLOW_TAXI_DAILY_TRIP_SUMMARY`
#### AUDIT LOGGING
- `NYCTAXI.AUDIT.PIPELINE_EXECUTION_LOG`

In [0]:
from datetime import datetime

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    TimestampType,
    DecimalType,
    DateType
)

# =====================================================
# WORKFLOW PARAMETERS
# =====================================================

dbutils.widgets.text("log_id", "")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("event_time", "")
dbutils.widgets.text("source_table", "")
dbutils.widgets.text("target_table", "")
dbutils.widgets.text("layer", "")
dbutils.widgets.text("notebook_path", "")
dbutils.widgets.text("pipeline_name", "")

# =====================================================
# RETRIEVE PARAMETERS
# =====================================================

log_id = dbutils.widgets.get("log_id")
run_id = dbutils.widgets.get("run_id")
raw_event_time = dbutils.widgets.get("event_time")

source_table = dbutils.widgets.get("source_table")
target_table = dbutils.widgets.get("target_table")
layer = dbutils.widgets.get("layer")

notebook_path = dbutils.widgets.get("notebook_path")
pipeline_name = dbutils.widgets.get("pipeline_name")

# =====================================================
# EVENT TIME
# =====================================================

try:
    if not raw_event_time or raw_event_time.startswith("{{"):
        event_time = datetime.now()
    else:
        event_time = datetime.fromisoformat(raw_event_time)
except Exception:
    event_time = datetime.now()

# =====================================================
# CURRENT USER
# =====================================================

user_name = spark.sql("SELECT current_user() AS user_name").first()["user_name"]

# =====================================================
# INITIALIZE AUDIT VARIABLES
# =====================================================

record_count = 0
status = "FAILED"
event_type = "LOAD_FAILURE"
message = ""

# =====================================================
# BUSINESS LOAD
# =====================================================

try:

    record_count = analytics_df.count()

    # Target Write
    analytics_df.write.mode("overwrite").saveAsTable("NYCTAXI.GOLD.YELLOW_TAXI_DAILY_TRIP_SUMMARY")

    status = "SUCCESS"
    event_type = "LOAD_SUCCESS"
    message = f"Loaded {record_count} records into {target_table}"

except Exception as e:

    status = "FAILED"
    event_type = "LOAD_FAILURE"
    message = str(e)

# =====================================================
# CAPTURE END TIME
# =====================================================

load_end_time = datetime.now()

# =====================================================
# AUDIT SCHEMA
# =====================================================

audit_schema = StructType([

    StructField("log_id", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("event_time", DateType(), True),
    StructField("event_type", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("record_count", LongType(), True),
    StructField("status", StringType(), True),
    StructField("message", StringType(), True),
    StructField("user_name", StringType(), True),
    StructField("notebook_path", StringType(), True),
    StructField("pipeline_name", StringType(), True),
    StructField("load_start_time", TimestampType(), True),
    StructField("load_end_time", TimestampType(), True),
    StructField("file_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("file_extension", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("source_folder", StringType(), True),
    StructField("file_size_bytes", LongType(), True),
    StructField("file_size_mb", DecimalType(18, 2), True),
    StructField("file_created_time", TimestampType(), True),
    StructField("file_modified_time", TimestampType(), True)

])

# =====================================================
# BUILD AUDIT RECORD
# =====================================================

audit_data = [(

    log_id,
    run_id,
    event_time,
    event_type,
    source_table,
    target_table,
    layer,
    record_count,
    status,
    message,
    user_name,
    notebook_path,
    pipeline_name,
    load_start_time,
    load_end_time,
    None,  # file_name
    None,  # file_path
    None,  # file_extension
    None,  # source_system
    None,  # source_folder
    None,  # file_size_bytes
    None,  # file_size_mb
    None,  # file_created_time
    None   # file_modified_time
)]

audit_df = spark.createDataFrame(
    audit_data,
    audit_schema
)

# =====================================================
# WRITE AUDIT LOG
# =====================================================

try:

    audit_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("NYCTAXI.AUDIT.PIPELINE_EXECUTION_LOG")

    print(f"Audit logging completed successfully for Run ID: {run_id}")

except Exception as audit_error:

    print(f"Business load completed but audit logging failed: {audit_error}")

# =====================================================
# FAIL NOTEBOOK IF LOAD FAILED
# =====================================================

if status == "FAILED":
    raise Exception(message)

In [0]:
dbutils.notebook.exit('YELLOW TAXI DAULY TRIP SUMMARY HAS BEEN LOADED INTO NYCTAXI.GOLD.YELLOW_TAXI_DAILY_TRIP_SUMMARY')